In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import scipy as sp
import scipy.signal as signal
import torchvision.transforms.functional as tvF
from sklearn import svm
from mpl_toolkits.axes_grid1 import make_axes_locatable

import math

In [ ]:
dtype = torch.float
# Check whether GPU is available
if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')


In [ ]:
def confusion_cal(y_test, py_test, y_type):
    class_num = len(np.unique(y_test))
    confusion_num = np.zeros((class_num, class_num))
    y_test = torch.tensor(y_test)
    py_test = torch.tensor(py_test)
    if y_type == 0:
        for ii in range(0, class_num):
            for jj in range(0, class_num):
                confusion_num[ii,jj] = len(np.intersect1d(torch.argwhere(torch.argmax(py_test, dim=1) == jj).detach().cpu().numpy(), torch.argwhere(y_test == ii).detach().cpu().numpy()))
    elif y_type == 1:
        for ii in range(0, class_num):
            for jj in range(0, class_num):
                confusion_num[ii,jj] = len(np.intersect1d(torch.argwhere(py_test == jj).detach().cpu().numpy(), torch.argwhere(y_test == ii).detach().cpu().numpy()))

    return confusion_num

In [ ]:
SPK = []
ptype = []
for rnum in range(0,28):
    SPK.append(np.load(f'/content/drive/MyDrive/Project/LearningTrajectories_separation/FixedRepresentation/data0/Spk_an_wn_{rnum}.npy'))
    ptype.append(np.load(f'/content/drive/MyDrive/Project/LearningTrajectories_separation/FixedRepresentation/data0/Ptype_an_wn_{rnum}.npy'))

SPK = np.concatenate(SPK, axis=0)
ptype = np.concatenate(ptype, axis=0)

SPK1 = []
ptype1 = []
for rnum in range(0,38):
    SPK1.append(np.load(f'/content/drive/MyDrive/Project/LearningTrajectories_separation/FixedRepresentation/data1/Spk_an_wn_{rnum}.npy'))
    ptype1.append(np.load(f'/content/drive/MyDrive/Project/LearningTrajectories_separation/FixedRepresentation/data1/Ptype_an_wn_{rnum}.npy'))

SPK1 = np.concatenate(SPK1, axis=0)
ptype1 = np.concatenate(ptype1, axis=0)

In [ ]:
train_ind = np.random.choice(np.size(SPK,axis=0), np.int32(0.8 * np.size(SPK,axis=0)), replace=False)
test_ind = np.setdiff1d(np.arange(0, np.size(SPK,axis=0)), train_ind)

In [ ]:
test_ind_od = np.random.choice(np.size(SPK1,axis=0), np.int32(0.2 * np.size(SPK1,axis=0)), replace=False)

In [ ]:
x_train = SPK[train_ind,:]
y_train = ptype[train_ind]

x_test = SPK[test_ind,:]
y_test = ptype[test_ind]

x_test_od = SPK1[test_ind_od,:]
y_test_od = ptype1[test_ind_od]

In [ ]:
acu_set = np.zeros((50,6))
acu_set_test = np.zeros((50,6))
for ii in range(0, 50):

    # Initialize model
    model = svm.LinearSVC()

    # Fit the model to training data
    model.fit(x_train, y_train)

    # Check test set accuracy
    acu_set[ii,0] = model.score(x_test, y_test)
    acu_set_test[ii,0] = model.score(x_test_od, y_test_od)

    for jj in range(0,5):
        acu_set[ii,jj + 1] = model.score(x_test[np.argwhere(y_test == jj).flatten()], y_test[np.argwhere(y_test == jj).flatten()])
        acu_set_test[ii,jj + 1] = model.score(x_test_od[np.argwhere(y_test_od == jj).flatten()], y_test_od[np.argwhere(y_test_od == jj).flatten()])


In [ ]:
np.save('/content/drive/MyDrive/Project/LearningTrajectories_separation/Result/svm_fixedrepresentation.npy', acu_set)
np.save('/content/drive/MyDrive/Project/LearningTrajectories_separation/Result/svm_fixedrepresentation1.npy', acu_set_test)
np.save('/content/drive/MyDrive/Project/LearningTrajectories_separation/Result/svm_fixedrepresentation_confusion_num.npy', confusion_cal(y_test, model.predict(x_test), y_type=1))
np.save('/content/drive/MyDrive/Project/LearningTrajectories_separation/Result/svm_fixedrepresentation_confusion_num1.npy', confusion_cal(y_test_od, model.predict(x_test_od), y_type=1))

In [ ]:
acu_set

array([[0.58709677, 0.68461538, 0.39495798, 0.41322314, 0.47540984,
        0.9375    ],
       [0.58709677, 0.68461538, 0.39495798, 0.41322314, 0.47540984,
        0.9375    ],
       [0.58709677, 0.68461538, 0.39495798, 0.41322314, 0.47540984,
        0.9375    ],
       [0.58709677, 0.68461538, 0.39495798, 0.41322314, 0.47540984,
        0.9375    ],
       [0.58709677, 0.68461538, 0.39495798, 0.41322314, 0.47540984,
        0.9375    ],
       [0.58709677, 0.68461538, 0.39495798, 0.41322314, 0.47540984,
        0.9375    ],
       [0.58709677, 0.68461538, 0.39495798, 0.41322314, 0.47540984,
        0.9375    ],
       [0.58709677, 0.68461538, 0.39495798, 0.41322314, 0.47540984,
        0.9375    ],
       [0.58709677, 0.68461538, 0.39495798, 0.41322314, 0.47540984,
        0.9375    ],
       [0.58709677, 0.68461538, 0.39495798, 0.41322314, 0.47540984,
        0.9375    ],
       [0.58709677, 0.68461538, 0.39495798, 0.41322314, 0.47540984,
        0.9375    ],
       [0.58709677, 0

In [ ]:
np.round(acu_set.mean(axis=0), 3)

array([0.587, 0.685, 0.395, 0.413, 0.475, 0.938])

In [ ]:
np.round(acu_set_test.mean(axis=0), 3)

array([0.604, 0.695, 0.475, 0.345, 0.487, 0.969])